In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from bboVisO.methods import angular, repro
from bboVisO import viso, opts, test
import numpy as np
from scipy.optimize import approx_fprime, check_grad
from calibcamlib import Camerasystem
import matplotlib.pyplot as plt

import jax

jax.config.update("jax_enable_x64", True)
%matplotlib ipympl

In [ ]:
np.random.seed(0)

viso_opts = opts.get_opts()
# precalc = viso.get_precalc(viso_opts)

calibs = Camerasystem.load_dict("/media/smb/soma-fs.ad01.caesar.de/bbo/projects/junker-bird/experiments/20230612_20230615_FlightWindow/calibrations/eyecams/20230615_0/multicam_calibration_viso_viso.yml")["calibs"]
landmarks = np.random.rand(5, 3)*20
positions = np.random.rand(7, 3)*10
oris = np.random.rand(7, 3)*np.pi
imcoords = np.random.rand(2, 7, 5, 2)

In [ ]:
opts.angular_jax_po_pol["optimize"][1]

In [ ]:
vars_free, objfun_args, optim_args = viso.make_optim_input(calibs, opts.angular_jax_po_pol, landmarks, positions, oris, 
                                                           landmark_imcoords=imcoords,
                                                           optimize=['positions', 'oris', 'landmarks', 'extrinsics_r', 'extrinsics_t'])

In [ ]:
objfun_angular_wrapper_lambda = lambda x: angular.objfun_wrapper(x, objfun_args)
objfun_repro_wrapper_lambda = lambda x: repro.objfun_wrapper(x, objfun_args)

In [ ]:
eps = 1e-10

repro_jax_dense_jac = repro.optfuncs.objfun_jacobian_wrapper_dense(vars_free, objfun_args)
repro_jax_sparse_jac = repro.optfuncs.objfun_jacobian_wrapper(vars_free, objfun_args).todense()
repro_approx_fprime_jac = approx_fprime(vars_free, repro.objfun_wrapper, eps, objfun_args)
repro_calc_2point_jacobian_jac = test.calc_2point_jacobian(repro.objfun_wrapper, vars_free, eps, objfun_args)
repro_sparsity_mask = repro.make_jacobian_sparsity_mask(*imcoords.shape[:3], objfun_args["mask_opt"])

angular_jax_dense_jac = angular.optfuncs.objfun_jacobian_wrapper_dense(vars_free, objfun_args)
angular_jax_sparse_jac = angular.optfuncs.objfun_jacobian_wrapper(vars_free, objfun_args).todense()
angular_approx_fprime_jac = approx_fprime(vars_free, angular.objfun_wrapper, eps, objfun_args)
angular_calc_2point_jacobian_jac = test.calc_2point_jacobian(angular.objfun_wrapper, vars_free, eps, objfun_args)
angular_sparsity_mask = angular.make_jacobian_sparsity_mask(*imcoords.shape[:3], objfun_args["mask_opt"])

In [ ]:
fig, axs = plt.subplots(2, 6, figsize=(22,8))
fig.set_tight_layout(True)
vmax = 100
im = axs[0, 0].imshow(repro_jax_dense_jac, vmin=0, vmax=vmax)
axs[0, 1].imshow(repro_jax_sparse_jac, vmin=0, vmax=vmax)
axs[0, 2].imshow(repro_approx_fprime_jac, vmin=0, vmax=vmax)
axs[0, 3].imshow(repro_calc_2point_jacobian_jac, vmin=0, vmax=vmax)
axs[0, 4].imshow(repro_sparsity_mask, vmin=0, vmax=1)
fig.colorbar(im, cax=axs[0, -1])

vmax = 0.9
im = axs[1, 0].imshow(angular_jax_dense_jac, vmin=0, vmax=vmax)
axs[1, 1].imshow(angular_jax_sparse_jac, vmin=0, vmax=vmax)
axs[1, 2].imshow(angular_approx_fprime_jac, vmin=0, vmax=vmax)
axs[1, 3].imshow(angular_calc_2point_jacobian_jac, vmin=0, vmax=vmax)
axs[1, 4].imshow(angular_sparsity_mask, vmin=0, vmax=1)
fig.colorbar(im, cax=axs[1, -1])


assert not np.any((np.abs(angular_jax_dense_jac)+np.abs(angular_approx_fprime_jac)+np.abs(angular_calc_2point_jacobian_jac))[~angular_sparsity_mask]), "Angular: Jacobian entries outside sparsity matrix"
assert not np.any((np.abs(repro_jax_dense_jac)+np.abs(repro_approx_fprime_jac)+np.abs(repro_calc_2point_jacobian_jac))[~repro_sparsity_mask]), "Repro: Jacobian entries outside sparsity matrix"
assert np.sum(np.abs(angular_jax_dense_jac-angular_jax_sparse_jac))==0, "Angular: Sparse jacobian does not match dense one"
assert np.sum(np.abs(repro_jax_dense_jac-repro_jax_sparse_jac))==0, "Repro: Sparse jacobian does not match dense one"

In [ ]:
from bboVisO.methods import angular, repro
from bboVisO import viso, opts, test
from calibcamlib import Camerasystem
import numpy as np
from time import time

np.random.seed(0)
m = 2
k = 1000
n = 50

calibs = Camerasystem.load_dict("/media/smb/soma-fs.ad01.caesar.de/bbo/projects/junker-bird/experiments/20230612_20230615_FlightWindow/calibrations/eyecams/20230615_0/multicam_calibration_viso_viso.yml")["calibs"]
landmarks = np.random.rand(n, 3)*20
positions = np.random.rand(k, 3)*10
oris = np.random.rand(k, 3)*np.pi
imcoords = np.random.rand(2, k, n, 2)

vars_free, objfun_args, optim_args = viso.make_optim_input(calibs, opts.angular_jax_po_pol, landmarks, positions, oris, 
                                                           landmark_imcoords=imcoords,
                                                           optimize=['positions', 'oris', 'landmarks', 'extrinsics_r', 'extrinsics_t'])


mask_opt = np.zeros_like(objfun_args["mask_opt"], dtype=bool)
for i in range(3):
    mask_opt |= np.random.random(mask_opt.shape)>0

angular.objfun_jacobian_wrapper(vars_free, objfun_args)
t = time()
angular.objfun_jacobian_wrapper(vars_free+1, objfun_args)
print(time()-t)